In [1]:
# Load environment variables and suppress warnings for clean output
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv()

print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


In [2]:
# Import LangChain components and initialize deterministic model
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5-mini", temperature=0)

print("Model initialized with gpt-5-mini.")

Model initialized with gpt-5-mini.


In [3]:
# Define realistic support ticket input
ticket = """
Users report that report exports are failing with error code EXPORT_402.
The issue started after the latest deployment.
Affects finance team only.
"""

In [4]:
# Define prompt without examples
prompt_no_examples = ChatPromptTemplate.from_messages([
    ("system", "You are a support analyst. Classify severity and summarize."),
    ("human",
     "Analyze this ticket:\n\n{ticket}\n\n"
     "Return exactly two lines:\n"
     "Severity: <Low|Medium|High>\n"
     "Summary: <one line>"
    )
])

chain_no_examples = prompt_no_examples | model

response_no_examples = chain_no_examples.invoke({"ticket": ticket}).content

print("=== OUTPUT WITHOUT EXAMPLES ===")
print(response_no_examples)

=== OUTPUT WITHOUT EXAMPLES ===
Severity: High
Summary: Report exports failing with error code EXPORT_402 after the latest deployment, preventing the finance team from exporting reports.


In [5]:
# Define prompt with embedded examples to guide structure and tone
prompt_with_examples = ChatPromptTemplate.from_messages([
    ("system", "You are a support analyst. Classify severity and summarize."),
    ("human",
     "Here are examples:\n\n"
     "Ticket: Login page loads slowly for all users.\n"
     "Severity: Medium\n"
     "Summary: Login performance degradation affecting users.\n\n"
     "Ticket: Database connection timeout across regions.\n"
     "Severity: High\n"
     "Summary: System-wide database connectivity failure.\n\n"
     "Now analyze this ticket:\n\n{ticket}\n\n"
     "Return exactly two lines:\n"
     "Severity: <Low|Medium|High>\n"
     "Summary: <one line>"
    )
])

chain_with_examples = prompt_with_examples | model

response_with_examples = chain_with_examples.invoke({"ticket": ticket}).content

print("\n=== OUTPUT WITH EXAMPLES ===")
print(response_with_examples)


=== OUTPUT WITH EXAMPLES ===
Severity: Medium
Summary: Report export failures (EXPORT_402) after latest deployment impacting finance team.


In [6]:
# Define prompt with stronger examples + clear decision pattern
prompt_with_examples = ChatPromptTemplate.from_messages([
    ("system",
     "You are a support analyst. Be strict and consistent.\n"
     "Use these rules:\n"
     "- High: outage, many users, core workflow blocked, or no workaround\n"
     "- Medium: partial impact, limited group, workaround exists\n"
     "- Low: minor issue, cosmetic, rare, or easy workaround\n"
     "Always write Summary as: Impact + scope + likely trigger."
    ),

    ("human",
     "Examples:\n\n"
     "Ticket: Export fails for all users in multiple departments. Started after deployment.\n"
     "Severity: High\n"
     "Summary: Exports blocked for many users after deployment change.\n\n"
     "Ticket: Billing page slow only for one customer. Workaround is using mobile app.\n"
     "Severity: Medium\n"
     "Summary: Billing workflow degraded for one customer; workaround available.\n\n"
     "Ticket: UI alignment issue in settings page. No functional impact.\n"
     "Severity: Low\n"
     "Summary: Cosmetic UI issue with no workflow impact.\n\n"
     "Now analyze this ticket:\n\n{ticket}\n\n"
     "Return exactly two lines:\n"
     "Severity: <Low|Medium|High>\n"
     "Summary: <one line>"
    )
])

chain_with_examples = prompt_with_examples | model
response_with_examples = chain_with_examples.invoke({"ticket": ticket}).content

print("\n=== OUTPUT WITH STRONGER EXAMPLES ===")
print(response_with_examples)


=== OUTPUT WITH STRONGER EXAMPLES ===
Severity: High
Summary: Exports blocked for finance team after deployment change.


In [7]:
# Reuse the same example-driven prompt with a new ticket (consistency wins)
ticket_2 = (
    "Settings page shows a misaligned icon in dark mode. "
    "No errors reported and workflow is unaffected."
)

out_with_examples_2 = chain_with_examples.invoke({"ticket": ticket_2}).content

print("=== WITH EXAMPLES (NEW TICKET) ===")
print(out_with_examples_2)

=== WITH EXAMPLES (NEW TICKET) ===
Severity: Low
Summary: Cosmetic misaligned icon on Settings page in dark mode affecting users in dark mode; likely caused by a dark-mode CSS/styling mismatch.
